# 04 Accelerating Code with Numba

[![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE)


In [ ]:
%config InlineBackend.figure_format = "retina"
# === Environment Setup ===
import random
import timeit
import numpy as np
from numba import njit, prange

# --- Configuration ---
# No plotting in this notebook, but setting up standard options
np.set_printoptions(suppress=True, linewidth=120, precision=4)



# The Lens: Compiling Python for Speed

**What economic problem are we solving?**
Economics often involves solving models that require millions of iterations—estimating a structural model via simulated method of moments, solving a dynamic programming problem with a large state space, or running Monte Carlo simulations for risk analysis. In pure Python, these loops are slow because the interpreter must check variable types at every step.

**Why do we need this method?**
**Numba** is a Just-In-Time (JIT) compiler that translates a subset of Python and NumPy code into fast machine code at runtime. It allows economists to write code that looks like Python but runs at the speed of C or Fortran. By adding a simple decorator (`@njit`), we can accelerate numerical loops by 100x or more, making previously intractable problems solvable on a laptop.

### Table of Contents
1.  [Introduction](#1.-Introduction)
2.  [A Canonical Example: The Monte Carlo Pi Simulation](#2.-A-Canonical-Example:-The-Monte-Carlo-Pi-Simulation)
    *   [Pure Python Implementation](#Pure-Python-Implementation)
    *   [Numba Implementation](#Numba-Implementation)
3.  [Numba with NumPy](#3.-Numba-with-NumPy)
4.  [Automatic Parallelization](#4.-Automatic-Parallelization)
5.  [When to Use Numba](#5.-When-to-Use-Numba)
6.  [Summary](#6.-Summary)

### 1. Introduction

Economic models, especially those involving simulation, optimization, or repeated estimation, are often computationally intensive. A common bottleneck is the execution speed of pure Python code. While Python is lauded for its readability and ease of use, it is an interpreted language, and its loops can be orders of magnitude slower than compiled languages like C, C++, or Fortran.

Traditionally, overcoming this involved complex workflows: writing performance-critical code in a low-level language, compiling it, and then writing Python "wrappers" to call it. This process is time-consuming and requires multi-language expertise.

**Numba** changes this paradigm. Numba is a **Just-In-Time (JIT) compiler** that translates a subset of Python and NumPy code into fast, native machine code. It allows you to achieve performance comparable to C or Fortran without ever leaving the Python ecosystem.

### 2. A Canonical Example: The Monte Carlo Pi Simulation

A simple way to demonstrate Numba's power is with a Monte Carlo simulation to estimate $\pi$. The logic is as follows:
1. Imagine a square with side length 2, centered at the origin. Its area is 4.
2. Inscribe a circle with radius 1 within this square. Its area is $\pi r^2 = \pi$.
3. Generate a large number of random points $(x, y)$ within the square.
4. The proportion of points that fall inside the circle should be equal to the ratio of the circle's area to the square's area: $\frac{\text{Points in Circle}}{\text{Total Points}} \approx \frac{\pi}{4}$.
5. Therefore, $\pi \approx 4 \times \frac{\text{Points in Circle}}{\text{Total Points}}$.

A point $(x, y)$ is inside the circle if $x^2 + y^2 < 1$. This requires a loop, which is notoriously slow in pure Python.

#### Pure Python Implementation

In [ ]:
def monte_carlo_pi_python(num_samples):
    """Estimates pi using a pure Python loop."""
    acc = 0
    for _ in range(num_samples):
        x = random.random()
        y = random.random()
        if (x**2 + y**2) < 1.0:
            acc += 1
    return 4.0 * acc / num_samples

num_samples = 10_000_000
py_time = timeit.timeit(lambda: monte_carlo_pi_python(num_samples), number=1)
print(f"> **Note:** Pure Python time: {py_time:.4f} seconds")

#### Numba Implementation

To accelerate this function with Numba, we simply apply the `@njit` decorator. `njit` stands for "no-python JIT," which is Numba's highest-performance compilation mode. It compiles the function so that it runs entirely without the involvement of the Python interpreter.

**Important:** Numba cannot compile all Python code. For example, it does not support the standard `random` module. We must use `np.random.rand()` inside the Numba-jitted function.

In [ ]:
@njit
def monte_carlo_pi_numba(num_samples):
    """Estimates pi using a Numba-compiled loop."""
    acc = 0
    for _ in range(num_samples):
        x = np.random.rand()
        y = np.random.rand()
        if (x**2 + y**2) < 1.0:
            acc += 1
    return 4.0 * acc / num_samples

# The first run compiles the function
print("Warming up Numba...")
monte_carlo_pi_numba(1)

# Now let's time it
numba_time = timeit.timeit(lambda: monte_carlo_pi_numba(num_samples), number=1)
print(f"Numba time:       {numba_time:.4f} seconds")
print(f"> **Note:** Speedup: {py_time / numba_time:.1f}x")

You should observe a speedup of 100x or more. This is the power of JIT compilation. You've achieved C-like speed with a single line of Python code.

### 3. Numba with NumPy

Numba is specifically designed to work well with NumPy arrays and functions. When Numba compiles code that uses NumPy arrays, it generates specialized, fast code that can operate directly on the underlying data buffers, avoiding the overhead of Python's object model.

In [ ]:
def sum_of_squares_python(arr):
    """Calculates the sum of squares of a NumPy array using a pure Python loop."""
    total = 0.0
    for i in range(arr.shape[0]):
        total += arr[i] ** 2
    return total

@njit
def sum_of_squares_numba(arr):
    """Calculates the sum of squares of a NumPy array using a Numba-compiled loop."""
    total = 0.0
    for i in range(arr.shape[0]):
        total += arr[i] ** 2
    return total

my_array = np.random.randn(10_000_000)

py_sum_time = timeit.timeit(lambda: sum_of_squares_python(my_array), number=1)
print(f"Python sum of squares time: {py_sum_time:.4f} seconds")

numba_sum_time = timeit.timeit(lambda: sum_of_squares_numba(my_array), number=1)
print(f"Numba sum of squares time:  {numba_sum_time:.4f} seconds")
print(f"> **Note:** Speedup: {py_sum_time / numba_sum_time:.1f}x")

### 4. Automatic Parallelization

Numba can also automatically parallelize some loops, allowing you to take advantage of multi-core CPUs with minimal effort. By adding the `parallel=True` argument to the decorator, you can instruct Numba to attempt to parallelize the function. You can then use `numba.prange` to mark loops that are safe to run in parallel.

In [ ]:
@njit(parallel=True)
def sum_of_squares_parallel(arr):
    """Calculates the sum of squares of a NumPy array using a Numba-compiled, parallel loop."""
    total = 0.0
    # prange indicates this loop can be parallelized
    for i in prange(arr.shape[0]):
        total += arr[i] ** 2
    return total

parallel_time = timeit.timeit(lambda: sum_of_squares_parallel(my_array), number=1)
print(f"Numba parallel time: {parallel_time:.4f} seconds")
print(f"> **Note:** Parallel Speedup: {numba_sum_time / parallel_time:.1f}x")

### 5. When to Use Numba

Numba is not a silver bullet. It works best on a specific type of problem:

- **Numerically-Oriented Code:** It is designed for numerical data types (integers, floats) and NumPy arrays.
- **Loops:** Numba's biggest advantage is in accelerating loops.
- **Avoid Unsupported Python Features:** Numba does not support all of Python. It does not work well with pandas DataFrames, dictionaries, or complex class structures. The best practice is to isolate your slow, looping, numerical code into a dedicated function and apply Numba to that function.

---
## Summary

In this lecture, we have systematically explored the theoretical and practical aspects of the model.

**Key Takeaways:**
1.  **Foundations:** We established the mathematical basis of the economic problem.
2.  **Computation:** We implemented the solution using efficient algorithms.
3.  **Implications:** We analyzed the economic significance of the results.

**Further Exploration:**
- Experiment with model parameters to assess sensitivity.
- Extend the framework by relaxing simplifying assumptions.